# Revizyona Kapalı Yeni Öncü Aday Taraması — Ders Kitabı Notebooku

Prompt 42'nin masa-başı karar yüzeyi. Veri indirme veya model çalıştırmaz; üç adayın as-of, mekanizma ve kapsam kapılarını görünür kılar.

## Okuma hedefleri

- Revizyona-kapalılık ile yalnız uzun tarih kapsamını ayırmak.
- Bir adayın gerçek `False` bilgi boşluğunu kapatıp kapatmadığını denetlemek.
- TCMB, SBM ve TÜİK kartlarının çoklu-kapı sonuçlarını karşılaştırmak.
- Altı aday ailesinden çıkan yapısal as-of bulgusunu doğru sınırla okumak.

In [1]:
import pandas as pd
from IPython.display import display, Markdown
kartlar = pd.DataFrame([
 {'aday':'TCMB kart işlem adedi','ilk':'2014-03','frekans':'haftalık','M−2':'evet','revizyona_kapali':'hayır','mekanizma':'kirli','kapsam':'yeterli','hukum':'ELENDI'},
 {'aday':'SBM trafik poliçe adedi','ilk':'2024','frekans':'aylık/kümülatif','M−2':'doğrulanmadı','revizyona_kapali':'doğrulanmadı/mutable','mekanizma':'yenileme karışık','kapsam':'N<50','hukum':'ELENDI'},
 {'aday':'TÜİK NACE45 satış hacmi','ilk':'2010-01','frekans':'aylık','M−2':'evet','revizyona_kapali':'hayır','mekanizma':'eşzamanlı/geniş','kapsam':'yeterli','hukum':'ELENDI'},
])
display(kartlar)

,aday,ilk,frekans,M−2,revizyona_kapali,mekanizma,kapsam,hukum
0,TCMB kart işlem adedi,2014-03,haftalık,evet,hayır,kirli,yeterli,ELENDI
1,SBM trafik poliçe adedi,2024,aylık/kümülatif,doğrulanmadı,doğrulanmadı/mutable,yenileme karışık,N<50,ELENDI
2,TÜİK NACE45 satış hacmi,2010-01,aylık,evet,hayır,eşzamanlı/geniş,yeterli,ELENDI


## Çoklu-kapı mantığı

TCMB: revizyon + mekanizma uyuşmazlığı. SBM: kısa kapsam + aktif mutable kayıt + yenileme/satış karışımı. TÜİK: revizyon + gerçek boşlukla zayıf eşleşme + eşzamanlılık. Tek güçlü özellik diğer kapıları telafi etmez.

In [2]:
kapilar = pd.DataFrame([
 ['TCMB',False,True,True,False],
 ['SBM',False,False,False,False],
 ['TÜİK',False,True,False,False],
], columns=['aday','revizyona_kapali','M−2_zamanli','N50_kapsam','temiz_bosluk_mekanizmasi']).set_index('aday')
display(kapilar)
assert not kapilar.all(axis=1).any()

,revizyona_kapali,M−2_zamanli,N50_kapsam,temiz_bosluk_mekanizmasi
aday,,,,
TCMB,False,True,True,False
SBM,False,False,False,False
TÜİK,False,True,False,False


## Model 13'ten genel metodolojik ders

BDDK taramasında delta marj C=1'den C=0,01'e `+0,2402 → +0,1091 → +0,0268` azaldı. Tek kapasite noktasındaki pozitif delta, mutlak marj ve kapasiteye göre null95 eğrisi olmadan yorumlanamaz.

In [3]:
kapasite = pd.DataFrame({'C':[1.0,0.1,0.01],'delta_marj':[0.2401713367,0.1091288246,0.0268479399],'kontrol_null95':[0.4683733695,0.4450343896,0.4219864222]})
display(kapasite)
assert kapasite['delta_marj'].is_monotonic_decreasing and kapasite['kontrol_null95'].is_monotonic_decreasing

,C,delta_marj,kontrol_null95
0,1.00,0.240171,0.468373
1,0.10,0.109129,0.445034
2,0.01,0.026848,0.421986


## Yapısal bulgu ve terminal hüküm

İki sınırlı taramada altı aile tüketildi: BDDK, BETAM–sahibindex, Google Trends, TCMB, SBM ve TÜİK. Bu örneklemde temiz hedef mekanizmasıyla kamuya açık ilk-yayım/as-of korunumu birlikte kurulamadı. Bu, tüm Türkiye verilerinin yokluğu değil; mevcut hedef ve as-of disiplini altında incelenen ailelerin sonucudur.

In [4]:
display(Markdown('**Hüküm:** `BU_TURDA_UYGUN_ADAY_YOK`  \n**İlerletilen aday:** `0/3`  \n**Sayfa bütçesi:** `7/10`  \n**Sonraki işlem:** kullanıcı yön kararı beklenir.'))

**Hüküm:** `BU_TURDA_UYGUN_ADAY_YOK`  
**İlerletilen aday:** `0/3`  
**Sayfa bütçesi:** `7/10`  
**Sonraki işlem:** kullanıcı yön kararı beklenir.

## Kullanıcı karar alanı

1. Seçenek 2'yi negatif as-of bulgusuyla kapat.
2. Bugünden başlayan ileriye dönük gölge vintaj arşivi kur.
3. Seçenek 3'e geçerek ufuk/toplulaştırma/sınıf sözleşmesini yeniden ele al.

Puanlama tablosu kullanıcı adına doldurulmaz; kilitli test açılmaz.